In [ ]:
# run this notebook after 1_curate_pca_variants_python
# use this notebook to run PCA on HGDP + 1KG
# after this, run 3_project_aou_python

In [ ]:
import os
import pandas as pd
bucket=os.getenv("WORKSPACE_BUCKET")
import hail as hl
hl.init(default_reference="GRCh38", tmp_dir = f'{bucket}/tmp/')

In [ ]:
mt_unrelated = hl.read_matrix_table("gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg_v2/pca_results/unrelateds_without_outliers.mt")
ht = mt_unrelated.rows()
ht = ht.select('locus', 'alleles')
ht.export(f'{bucket}/pca/gnomad_nb2_loci_alleles.tsv')

In [ ]:
pca_variants_final_ht = hl.import_table(f'{bucket}/pca/pca_variants_acaf.tsv')
pca_variants_final_ht = pca_variants_final_ht.annotate(
    locus = hl.locus(pca_variants_final_ht.chr, hl.int(pca_variants_final_ht.pos), reference_genome='GRCh38'),
    alleles = hl.parse_json(pca_variants_final_ht.final_alleles, hl.tarray(hl.tstr)))
pca_variants_final_ht = pca_variants_final_ht.key_by('locus', 'alleles')

In [ ]:
mt = hl.read_matrix_table("gs://gcp-public-data--gnomad/release/3.1.2/mt/genomes/gnomad.genomes.v3.1.2.hgdp_1kg_subset_dense.mt")
sample_ids_unrelated = mt_unrelated.cols().select().key_by('s')  # Just sample IDs
mt_filtered = mt.filter_cols(hl.is_defined(sample_ids_unrelated[mt.s]))
mt_filtered = mt_filtered.semi_join_rows(pca_variants_final_ht)
mt_filtered = mt_filtered.repartition(500)
mt_filtered.write(f'{bucket}/pca/subset_hgdp_1kg_unrelated_outliers_removed_acaf.mt', overwrite=True)

In [ ]:
eigenvalues, scores, loadings = hl.hwe_normalized_pca(mt_filtered.GT, k=20, compute_loadings=True)
loadings.export(f'{bucket}/pca/hgdp_1000g_variant_loadings_acaf.tsv')
scores.export(f'{bucket}/pca/hgdp_1000g_scores_acaf.tsv')
df = pd.DataFrame(eigenvalues)
df.to_csv(f'{bucket}/pca/hgdp_1000g_eigenvalues_acaf.tsv', sep='\t', index=False)